<a href="https://colab.research.google.com/github/GillValenzuela/curso_data_science/blob/master/DS_Ingemat_Clase_24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q --upgrade datasets fsspec==2025.3.0 \
    gcsfs==2025.3.0 --quiet
!pip install -q transformers datasets tqdm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 18.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-nvrtc-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-runtime-cu12==12.4.127; platform_system == "Linux" and platform_m

In [2]:
# ===========================================================
# Self-Attention (single layer, multi-head)  —  from scratch
# ===========================================================
import torch
import torch.nn.functional as F

# ----- 1. Mini input ----------------------------------------------------
# 4 tokens, embedding dim d_model = 8   (tiny so we can print everything)
x = torch.tensor([[ 1., 0., 0., 1., 0., 0., 1., 0.],
                  [ 0., 2., 0., 1., 0., 1., 0., 0.],
                  [ 0., 0., 3., 1., 0., 0., 0., 1.],
                  [ 1., 1., 1., 1., 1., 1., 1., 1.]], requires_grad=False)   # (T=4, d=8)

T, d_model = x.shape
n_heads     = 2
d_head      = d_model // n_heads    # 4

# ----- 2. Weight matrices (tiny, deterministic) -------------------------
torch.manual_seed(0)
W_q = torch.randn(n_heads, d_model, d_head)   # (h, d, d_h)
W_k = torch.randn(n_heads, d_model, d_head)
W_v = torch.randn(n_heads, d_model, d_head)
W_o = torch.randn(d_model, d_model)

# ----- 3. Multi-head Attention -----------------------------------------
def multi_head_self_attention(x):
    heads_out = []
    for h in range(n_heads):
        Q = x @ W_q[h]                       # (T, d_h)
        K = x @ W_k[h]                       # (T, d_h)
        V = x @ W_v[h]                       # (T, d_h)

        scores  = Q @ K.T / (d_head ** 0.5)  # scaled dot-product (T,T)
        weights = F.softmax(scores, dim=-1)  # α_ij
        z_h     = weights @ V               # (T, d_h)
        heads_out.append(z_h)

        print(f"\nHead {h+1} — attention weights (rows sum→1):\n", weights)

    z = torch.cat(heads_out, dim=-1)         # concat (T, d)
    return z @ W_o                           # final linear proj (T, d)

out = multi_head_self_attention(x)
print("\nOutput Z (after W_o):\n", out)


Head 1 — attention weights (rows sum→1):
 tensor([[7.2314e-02, 8.3168e-02, 7.9486e-01, 4.9663e-02],
        [6.0056e-01, 3.9283e-01, 7.5931e-04, 5.8442e-03],
        [7.1833e-01, 2.7063e-01, 9.3617e-03, 1.6792e-03],
        [1.9565e-02, 2.1619e-03, 9.7818e-01, 8.9041e-05]])

Head 2 — attention weights (rows sum→1):
 tensor([[4.1017e-04, 8.6431e-04, 9.9863e-01, 1.0002e-04],
        [1.3702e-01, 7.1069e-04, 8.6103e-01, 1.2453e-03],
        [3.2507e-02, 2.2555e-01, 7.4180e-01, 1.4491e-04],
        [7.2638e-04, 3.5192e-04, 9.9835e-01, 5.7205e-04]])

Output Z (after W_o):
 tensor([[-1.9879,  0.1686, -0.7718, -1.9693,  1.2754, -2.0369, -2.2103,  2.1430],
        [ 0.4765,  3.5429, -1.1239, -0.6233,  1.0973,  0.2621, -5.7436,  2.4156],
        [-1.1835,  4.0991, -1.7781, -2.1197,  2.4349, -0.0596, -4.6619,  2.5935],
        [-2.7765, -0.6497, -0.9108, -2.6100,  1.6515, -2.6864, -1.4257,  2.4519]])


In [3]:
import torch
from transformers import BertModel, BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

text = "BERT's attention mechanism is fascinating."
inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
outputs = model(**inputs, output_attentions=True)

attention_weights = outputs.attentions
print(attention_weights)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


(tensor([[[[7.0430e-02, 4.7734e-02, 4.5779e-02,  ..., 9.7265e-02,
           1.3927e-01, 3.5823e-01],
          [3.6679e-02, 8.6646e-02, 4.3685e-02,  ..., 2.3391e-01,
           3.7266e-02, 6.5047e-02],
          [8.5532e-02, 1.3269e-01, 3.7626e-02,  ..., 1.2145e-01,
           1.1870e-01, 1.3301e-01],
          ...,
          [5.4352e-02, 1.2452e-01, 1.4481e-01,  ..., 1.2920e-01,
           1.0262e-01, 1.0715e-01],
          [8.2241e-02, 1.0184e-01, 9.4826e-02,  ..., 9.6448e-02,
           1.5859e-01, 1.1109e-01],
          [9.6005e-02, 6.8182e-02, 6.3413e-02,  ..., 1.3608e-01,
           1.7070e-01, 1.8861e-01]],

         [[6.8129e-01, 9.2499e-03, 2.2069e-01,  ..., 5.5413e-03,
           3.5570e-02, 7.2794e-03],
          [2.3929e-01, 5.2402e-02, 8.9530e-02,  ..., 1.1060e-01,
           1.2288e-01, 1.3781e-01],
          [1.9622e-02, 1.6688e-01, 1.2318e-02,  ..., 2.9411e-01,
           1.1144e-01, 3.4867e-02],
          ...,
          [1.6697e-01, 9.8040e-02, 3.0382e-02,  ..., 7.042

In [4]:
# ===============================================================
# Fine-tune BERT-base (encoder-only) en IMDB usando **puro PyTorch**
# ===============================================================
# 1) Instala dependencias si aún no están (≈1 min en Colab):
# !pip install -q transformers datasets tqdm

import torch, torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from tqdm.auto import tqdm
import numpy as np, random, os

In [5]:


# --------------------
# 0. Configuración
# --------------------
MODEL_NAME   = "bert-base-uncased"
MAX_LEN      = 256       # recorta / pad a 256 tokens
BATCH_TRAIN  = 8
BATCH_EVAL   = 16
EPOCHS       = 2
SEED         = 42

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando:", device)



Usando: cuda


In [6]:

# --------------------
# 1. Dataset IMDB
# --------------------
ds = load_dataset("imdb")                            # train=25k, test=25k
tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize(batch):
    return tok(batch["text"],
               truncation=True,
               padding="max_length",
               max_length=MAX_LEN)

ds_tok = ds.map(tokenize, batched=True, remove_columns=["text"])
ds_tok.set_format(type="torch",
                  columns=["input_ids", "attention_mask", "label"])

train_ds = ds_tok["train"].shuffle(SEED)[:20000]     # sub-sample p/ rapidez
val_ds   = ds_tok["test"] [:5000]

README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [7]:

# --------------------
# 2. DataLoaders
# --------------------
train_ds = ds_tok["train"].shuffle(seed=SEED).select(range(20_000))
val_ds   = ds_tok["test"].select(range(5_000))

train_dl = DataLoader(train_ds, batch_size=BATCH_TRAIN, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_EVAL,  shuffle=False)

In [8]:


# --------------------
# 3. Modelo
# --------------------
model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=2).to(device)

# Optimizer + LR scheduler warm-up lineal (simple)
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=1e-2)

total_steps = len(train_dl) * EPOCHS
scheduler = torch.optim.lr_scheduler.LinearLR(
              optimizer, start_factor=1.0, end_factor=0.0, total_iters=total_steps)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:

# --------------------
# 4. Función evaluación
# --------------------
def evaluate(model, loader):
    model.eval(); correct=total=0
    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y    = batch["label"].to(device)
            logits = model(ids, attention_mask=mask).logits
            preds  = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total   += y.size(0)
    return correct / total

In [10]:


# --------------------
# 5. Entrenamiento
# --------------------
for ep in range(1, EPOCHS+1):
    model.train()
    loop = tqdm(train_dl, desc=f"Epoch {ep}/{EPOCHS}", leave=False)
    for step, batch in enumerate(loop, 1):
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        y    = batch["label"].to(device)

        outputs = model(ids, attention_mask=mask, labels=y)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()

        if step % 100 == 0:
            loop.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")

    val_acc = evaluate(model, val_dl)
    print(f"Epoch {ep}:  valid accuracy = {val_acc*100:.2f}%")

# --------------------
# 6. Guardar modelo
# --------------------
os.makedirs("bert-imdb", exist_ok=True)
model.save_pretrained("bert-imdb")
tok.save_pretrained("bert-imdb")
print("✅ Modelo guardado en carpeta  bert-imdb")

Epoch 1/2:   0%|          | 0/2500 [00:00<?, ?it/s]

Epoch 1:  valid accuracy = 91.52%


Epoch 2/2:   0%|          | 0/2500 [00:00<?, ?it/s]

Epoch 2:  valid accuracy = 92.06%
✅ Modelo guardado en carpeta  bert-imdb
